In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve

In [2]:
plt.rcParams.update({"font.family": ["Calibri", "DejaVu Sans"], "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
RED = "#C0392B"

In [ ]:
df = pd.read_csv("../data/clean/user_features.csv", index_col="user_id")
df['age'] = df['age'].fillna(df['age'].median())
y = df["is_churn"]
X = df.drop(columns=["is_churn"]) 
X

In [5]:
# Feature selection: Remove Only highly correlated numerical features (|r| > 0.85)

corr = X.select_dtypes("number").corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))     
pairs = [(a, b, round(upper.loc[a, b], 2)) for a in upper.index for b in upper.columns if upper.loc[a, b] > 0.85]
to_drop = [c for c in upper.columns if (upper[c] > 0.85).any()]       
print("Pasangan berkorelasi > 0.85:", pairs)
print("Fitur dibuang:", to_drop)
X = X.drop(columns=to_drop)

Pasangan berkorelasi > 0.85: [('n_trx', 'total_amount', np.float64(0.88)), ('total_sessions', 'active_days', np.float64(0.91))]
Fitur dibuang: ['total_amount', 'active_days']


In [ ]:
# Preprocessing

num_cols = X.select_dtypes("number").columns.tolist()          # numeric: if still exsist null value, fill with median
cat_cols = X.select_dtypes(exclude="number").columns.tolist()  # categoric (teks): one-hot encoding
prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), num_cols),
                          ("cat", OneHotEncoder(), cat_cols)], verbose_feature_names_out=False)

def make_model(depth):
    return Pipeline([("prep", prep), ("tree", DecisionTreeClassifier(max_depth=depth, min_samples_leaf=10, random_state=42))])

In [ ]:
rep = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)
rows = []
for depth in range(1, 6):
    s = cross_val_score(make_model(depth), X, y, cv=rep, scoring="roc_auc")
    rows.append([depth, s.mean(), s.std()])
res = pd.DataFrame(rows, columns=["max_depth", "AUC", "AUC_std"])
print("\nAUC per depth tree:\n", res.round(3).to_string(index=False))
best = res.loc[res.AUC.idxmax()]
depth = int(res[res.AUC >= best.AUC - best.AUC_std].max_depth.min())   # selisih < 1 sd = sama baiknya -> pilih paling sederhana
print("depth choosen:", depth)


AUC per kedalaman pohon:
  max_depth   AUC  AUC_std
         1 0.644    0.059
         2 0.635    0.064
         3 0.640    0.064
         4 0.630    0.077
         5 0.626    0.078
Kedalaman terpilih: 1


In [ ]:
# Evaluation

sc = cross_validate(make_model(depth), X, y, cv=rep, scoring=["accuracy", "precision", "recall", "roc_auc"])
print(f"\nAccuracy : {sc['test_accuracy'].mean():.3f} ± {sc['test_accuracy'].std():.3f}   (guess 'all active' = {1 - y.mean():.3f})")
print(f"Precision: {sc['test_precision'].mean():.3f} ± {sc['test_precision'].std():.3f}")
print(f"Recall   : {sc['test_recall'].mean():.3f} ± {sc['test_recall'].std():.3f}")
auc = sc["test_roc_auc"].mean()
print(f"AUC      : {auc:.3f} ± {sc['test_roc_auc'].std():.3f}")

cv1 = StratifiedKFold(5, shuffle=True, random_state=42)                 
pred = cross_val_predict(make_model(depth), X, y, cv=cv1)
print("Confusion matrix (5-fold, 250 user):\n", confusion_matrix(y, pred))


Accuracy : 0.757 ± 0.045   (tebakan 'semua aktif' = 0.704)
Precision: 0.672 ± 0.143
Recall   : 0.366 ± 0.116
AUC      : 0.644 ± 0.059
Confusion matrix (5-fold, 250 user):
 [[162  14]
 [ 47  27]]


In [ ]:
model = make_model(depth).fit(X, y)
names = model.named_steps["prep"].get_feature_names_out()
tree = model.named_steps["tree"]
print("\nTree rules:\n", export_text(tree, feature_names=list(names)))
leaf = pd.DataFrame({"leaf": tree.apply(model.named_steps["prep"].transform(X)), "churn": y.values}).groupby("leaf").churn.agg(users="size", churners="sum", churn_rate="mean")
print("Churn rate per group (leaf tree):\n", leaf.round(3))
imp = pd.Series(tree.feature_importances_, index=names)
print("\nFeature importance:\n", imp[imp > 0].round(3))



Aturan pohon:
 |--- n_trx <= 1.50
|   |--- class: 1
|--- n_trx >  1.50
|   |--- class: 0

Churn rate per kelompok (daun pohon):
       users  churners  churn_rate
leaf                             
1        39        27       0.692
2       211        47       0.223

Feature importance (hanya yang dipakai):
 n_trx    1.0
dtype: float64


In [ ]:
# Visualization

nama = {"n_trx": "Total transaction", "trx_span_days": "Range transaction days\n(pertama-terakhir)"}
fig, ax = plt.subplots(figsize=(10, 5))
plot_tree(tree, feature_names=[nama.get(c, c) for c in names], class_names=["Active", "Churn"],
          filled=True, rounded=True, impurity=False, fontsize=13, ax=ax)
ax.set_title("Decision Tree: A short transaction history distinguishes users at risk of churn.", loc="left", fontweight="bold", fontsize=14)
plt.tight_layout(); plt.savefig("charts/6_decision_tree.png", dpi=150); plt.close()

fig, axs = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=["Active", "Churn"]).plot(ax=axs[0], cmap="Blues", colorbar=False)
axs[0].set_title("Confusion matrix (250 user)", fontweight="bold"); axs[0].grid(False)
fpr, tpr, _ = roc_curve(y, pred)
axs[1].plot(fpr, tpr, color=RED, lw=2.5, marker="o", label=f"Decision Tree (AUC = {auc:.2f})")
axs[1].plot([0, 1], [0, 1], "--", color="grey", label="Random guess (AUC = 0.50)")
axs[1].set_xlabel("False positive rate"); axs[1].set_ylabel("True positive rate"); axs[1].legend(frameon=False, loc="lower right")
axs[1].set_title("ROC Curve", fontweight="bold")
plt.tight_layout(); plt.savefig("charts/7_performa_model.png", dpi=150); plt.close()